# Gene Selection Method Evaluation

각 method가 반환한 gene set을 동일한 3가지 기준으로 평가:

| 평가 | 방법 | 지표 |
|---|---|---|
| Unsupervised | Silhouette (euclidean) + kNN purity (k=15) | 질병별 분리도 |
| Binary | HC vs Disease  5-fold CV LogReg + RF | 각 질병의 HC 구별 능력 |
| Multiclass | 20-class OVR  5-fold CV LogReg | 질병 간 구별 능력 |

`SELECTORS` dict에 함수를 추가하면 동일 평가 loop 적용됨.

In [32]:
import sys, warnings
warnings.filterwarnings('ignore')
sys.path.insert(0, '.')

from pathlib import Path
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.sparse import issparse
from sklearn.metrics import silhouette_samples, roc_auc_score, roc_curve
from sklearn.neighbors import NearestNeighbors
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.feature_selection import f_classif, mutual_info_classif
from sklearn.preprocessing import LabelEncoder
from umap import UMAP
from matplotlib.patches import Patch
import scanpy as sc

from model_engine import NormativeModelEngine

BASE_DIR    = Path('.').resolve()
DATA_DIR    = BASE_DIR.parent / 'OpenAccess_nfcore'
H5AD_PATH   = DATA_DIR / 'Merged_Processed_AnnData_with_Batch_Biases_QC_Status.h5ad'
ENGINE_DIR  = BASE_DIR / 'engine_state'
RESULTS_DIR = BASE_DIR / 'CV_Results'
FIG_DIR     = RESULTS_DIR / 'Figures'
FIG_DIR.mkdir(parents=True, exist_ok=True)

BIAS_COLUMNS = [
    'log(Total Reads)', 'Spliced Reads (%)',
    'gDNA Contamination (Intron/Exon)', 'rRNA Fraction',
    "RNA Degradation (3' Bias)", 'Platelet Score',
    'GC Bias', 'Gene Length Bias', 'NG80', '(NP80/NG80)',
]


In [33]:
# disease_scoring.ipynb에서 생성한 Z matrix 로드
Z_dis      = np.load(RESULTS_DIR / 'Z_disease.npy')                      # (n_dis, n_genes)
dis_names  = np.load(RESULTS_DIR / 'Z_sample_names.npy', allow_pickle=True).tolist()
gene_names = np.load(RESULTS_DIR / 'Z_gene_names.npy',   allow_pickle=True).tolist()
gn_arr     = np.array(gene_names)

n_nan = np.isnan(Z_dis).sum() + np.isinf(Z_dis).sum()
Z_dis = np.nan_to_num(Z_dis.astype(np.float32), nan=0.0, posinf=10.0, neginf=-10.0)
print(f'Z_dis: {Z_dis.shape}  (cleaned {n_nan:,} NaN/Inf → 0)')

print('Loading h5ad metadata...')
adata    = sc.read_h5ad(H5AD_PATH)
obs_mask = (
    (adata.obs['QC_Passed'] == True) &
    (adata.obs['Phenotype_Processed'].notna()) &
    (adata.obs['Phenotype_Processed'] != 'Unknown') &
    (adata.obs['broad_protocol_category'] != 'Exome-based (EB)')
)
adata     = adata[obs_mask, adata.var['GeneType'] == 'protein_coding'].copy()
is_hc     = (adata.obs['Phenotype_Processed'].astype(str) == 'Healthy Control').values
is_dis    = ~is_hc
Y         = np.round((adata.X.toarray() if issparse(adata.X) else np.asarray(adata.X))).astype(np.float32)
X_raw     = adata.obs[BIAS_COLUMNS].values.astype(np.float64)
phenos    = adata.obs['Phenotype_Processed'].astype(str).values
dis_pheno = phenos[is_dis]

print(f'phenotypes: {len(np.unique(dis_pheno))}')
print(pd.Series(dis_pheno).value_counts().to_string())


Z_dis: (913, 20097)  (cleaned 510,367 NaN/Inf → 0)
Loading h5ad metadata...
phenotypes: 20
CDCS                  224
Tuberculosis          103
ME/CFS                 90
Pancreatitis           81
Pancreatic Cancer      74
Pre-eclampsia          62
Liver Cancer           48
Colorectal Cancer      41
Lung Cancer            33
Stomach Cancer         29
Esophagus Cancer       27
MM                     18
Other Cancer           18
HIV                    13
HIV + Tuberculosis     11
ICI-m                  11
ICI-treated Cancer     11
MGUS                    8
Pancreatic Cancer       6
Liver Cirrhosis         5


In [34]:
hc_z_path  = RESULTS_DIR / 'Z_hc.npy'
hc_sn_path = RESULTS_DIR / 'Z_hc_names.npy'

if hc_z_path.exists():
    Z_hc     = np.load(hc_z_path)
    hc_names = np.load(hc_sn_path, allow_pickle=True).tolist()
    print(f'Z_hc loaded from cache: {Z_hc.shape}')
else:
    engine   = NormativeModelEngine.load(ENGINE_DIR)
    hc_idx   = np.where(is_hc)[0]
    hc_names = [adata.obs_names[i] for i in hc_idx]
    print(f'Scoring {len(hc_idx)} HC samples ...')
    t0   = time.perf_counter()
    res  = engine.score(X_raw[is_hc], Y[is_hc], gene_names=gene_names, seed=42)
    Z_hc = res['combined']                               # (n_hc, n_genes)
    np.save(hc_z_path,  Z_hc)
    np.save(hc_sn_path, np.array(hc_names))
    print(f'Saved Z_hc: {Z_hc.shape}  ({time.perf_counter()-t0:.0f}s)')

n_nan = np.isnan(Z_hc).sum() + np.isinf(Z_hc).sum()
Z_hc  = np.nan_to_num(Z_hc.astype(np.float32), nan=0.0, posinf=10.0, neginf=-10.0)
print(f'Z_hc cleaned: {n_nan:,} NaN/Inf → 0')


Z_hc loaded from cache: (693, 20097)
Z_hc cleaned: 387,387 NaN/Inf → 0


In [48]:
from scipy.linalg import svd
def _idx(genes):
    gmap = {g: i for i, g in enumerate(gene_names)}
    return np.array([gmap[g] for g in genes if g in gmap], dtype=int)

def select_proportion(n_per_pheno=30, thr=3.0):
    """Top-N per phenotype by proportion of samples with |z| >= thr."""
    flagged  = np.abs(Z_dis) >= thr
    selected = set()
    for ph in np.unique(dis_pheno):
        m    = dis_pheno == ph
        prop = flagged[m].mean(axis=0)
        selected.update(gn_arr[np.argsort(-prop)[:n_per_pheno]])
    return sorted(selected)

def select_effect_size(n_per_pheno=30):
    """Top-N per phenotype by mean |z| in that phenotype (HC baseline = 0)."""
    selected = set()
    for ph in np.unique(dis_pheno):
        m = dis_pheno == ph
        selected.update(gn_arr[np.argsort(-np.abs(Z_dis[m]).mean(axis=0))[:n_per_pheno]])
    return sorted(selected)

def select_svd_signature(n_per_pheno=30):
    selected = set()
    phenotypes = np.unique(dis_pheno)
    centroids = np.array([Z_dis[dis_pheno == ph].mean(axis=0) for ph in phenotypes])
    U, S, Vh = svd(centroids, full_matrices=False)
    for idx in range(len(phenotypes)):
        phenotype_direction = U[idx, :] @ np.diag(S) @ Vh
        top_indices = np.argsort(-np.abs(phenotype_direction))[:n_per_pheno]
        selected.update(gn_arr[top_indices])
        
    return sorted(selected)

# ── SELECTORS: 여기서 추가/수정 ──────────────────────────────────────
SELECTORS = {
    'proportion_top30':  lambda: select_proportion(n_per_pheno=30),
    'effect_size_top30': lambda: select_effect_size(n_per_pheno=30),
    'svd_top30': lambda: select_svd_signature(n_per_pheno=30),
}
print('Selectors:', list(SELECTORS.keys()))

Selectors: ['proportion_top30', 'effect_size_top30', 'svd_top30']


In [49]:
CV = 5
K  = 15
BASE_FPR = np.linspace(0, 1, 101)   # 공통 FPR grid (ROC shadow용)


def eval_unsupervised(idx):
    X   = Z_dis[:, idx]
    sil = silhouette_samples(X, dis_pheno, metric='euclidean')
    nbrs = NearestNeighbors(n_neighbors=K+1, metric='euclidean', n_jobs=-1).fit(X)
    nn   = nbrs.kneighbors(X)[1][:, 1:]
    pur  = np.array([np.mean(dis_pheno[nn[i]] == dis_pheno[i]) for i in range(len(dis_pheno))])
    rows = [{'phenotype': ph,
             'silhouette': float(sil[dis_pheno==ph].mean()),
             'knn_purity': float(pur[dis_pheno==ph].mean())}
            for ph in np.unique(dis_pheno)]
    return pd.DataFrame(rows).set_index('phenotype'), float(sil.mean())


def eval_binary(idx):
    """HC vs Disease per phenotype — 5-fold CV, LogReg + RF.
    Returns: (per-pheno AUC DataFrame, roc_dict)
    roc_dict[ph] = {'lr': {'fprs': [...], 'tprs': [...]}, 'rf': {...}}
                   None if n < CV (too few samples)
    """
    Xhc  = Z_hc[:, idx]
    skf  = StratifiedKFold(n_splits=CV, shuffle=True, random_state=42)
    lr   = LogisticRegression(max_iter=500, C=0.1)
    rf   = RandomForestClassifier(n_estimators=30, max_depth=6, random_state=42, n_jobs=-1)
    rows, roc_dict = [], {}

    for ph in np.unique(dis_pheno):
        m = dis_pheno == ph
        if m.sum() < CV:          # CV 미만이면 StratifiedKFold 불가
            rows.append({'phenotype': ph, 'auc_logreg': np.nan, 'auc_rf': np.nan})
            roc_dict[ph] = None
            continue

        X = np.vstack([Z_dis[m][:, idx], Xhc])
        y = np.array([1]*m.sum() + [0]*len(Xhc))
        lr_a, rf_a = [], []
        lr_roc = {'fprs': [], 'tprs': []}
        rf_roc = {'fprs': [], 'tprs': []}

        for tr, te in skf.split(X, y):
            try:
                lr.fit(X[tr], y[tr]); p = lr.predict_proba(X[te])[:, 1]
                fpr, tpr, _ = roc_curve(y[te], p)
                lr_roc['fprs'].append(fpr); lr_roc['tprs'].append(tpr)
                lr_a.append(roc_auc_score(y[te], p))

                rf.fit(X[tr], y[tr]); p = rf.predict_proba(X[te])[:, 1]
                fpr, tpr, _ = roc_curve(y[te], p)
                rf_roc['fprs'].append(fpr); rf_roc['tprs'].append(tpr)
                rf_a.append(roc_auc_score(y[te], p))
            except Exception:
                pass

        roc_dict[ph] = {'lr': lr_roc, 'rf': rf_roc}
        rows.append({'phenotype': ph,
                     'auc_logreg': float(np.mean(lr_a)) if lr_a else np.nan,
                     'auc_rf':     float(np.mean(rf_a)) if rf_a else np.nan})

    return pd.DataFrame(rows).set_index('phenotype'), roc_dict


def eval_multiclass(idx):
    """All-disease OVR multiclass — 5-fold CV LogReg.
    Returns: (per-pheno AUC DataFrame, macro_auc, roc_dict)
    roc_dict[ph] = {'fprs': [...], 'tprs': [...]}  (per-fold)
    """
    le   = LabelEncoder()
    y    = le.fit_transform(dis_pheno)
    X    = Z_dis[:, idx]
    skf  = StratifiedKFold(n_splits=CV, shuffle=True, random_state=42)
    clf  = LogisticRegression(max_iter=500, C=0.1, multi_class='ovr')

    fold_yt, fold_yp = [], []
    for tr, te in skf.split(X, y):
        clf.fit(X[tr], y[tr])
        fold_yt.append(y[te])
        fold_yp.append(clf.predict_proba(X[te]))

    rows, roc_dict = [], {}
    for i, ph in enumerate(le.classes_):
        class_roc = {'fprs': [], 'tprs': []}
        aucs = []
        for yt, yp in zip(fold_yt, fold_yp):
            try:
                y_bin = (yt == i).astype(int)
                fpr, tpr, _ = roc_curve(y_bin, yp[:, i])
                class_roc['fprs'].append(fpr); class_roc['tprs'].append(tpr)
                aucs.append(roc_auc_score(y_bin, yp[:, i]))
            except Exception:
                pass
        roc_dict[ph] = class_roc
        rows.append({'phenotype': ph, 'auc_multiclass': float(np.mean(aucs)) if aucs else np.nan})

    df = pd.DataFrame(rows).set_index('phenotype')
    return df, float(df['auc_multiclass'].mean()), roc_dict


print(f'Eval functions ready.  CV={CV}  K={K}')
print('Note: min binary samples = CV =', CV, '(Liver Cirrhosis n=5 포함)')


Eval functions ready.  CV=5  K=15
Note: min binary samples = CV = 5 (Liver Cirrhosis n=5 포함)


In [50]:
all_results = {}

for name, selector in SELECTORS.items():
    t0 = time.perf_counter()
    print(f'\n── {name} ──')
    genes = selector()
    idx   = _idx(genes)
    print(f'  {len(genes)} genes selected')

    df_unsup, macro_sil        = eval_unsupervised(idx);        print('  unsup done')
    df_bin,   roc_bin          = eval_binary(idx);               print('  binary done')
    df_mc,    macro_mc, roc_mc = eval_multiclass(idx);           print('  multiclass done')

    per_pheno = df_unsup.join(df_bin, how='left').join(df_mc, how='left')
    per_pheno.insert(0, 'n', pd.Series(dis_pheno).value_counts())

    all_results[name] = dict(
        genes     = genes,
        per_pheno = per_pheno,
        macro_sil = macro_sil,
        macro_mc  = macro_mc,
        roc_bin   = roc_bin,
        roc_mc    = roc_mc,
        n_genes   = len(genes),
        t         = time.perf_counter() - t0,
    )
    print(f'  sil={macro_sil:.3f}  mc_auc={macro_mc:.3f}  ({all_results[name]["t"]:.0f}s)')

print('\nAll selectors done.')



── proportion_top30 ──
  537 genes selected
  unsup done
  binary done
  multiclass done
  sil=-0.005  mc_auc=0.948  (11s)

── effect_size_top30 ──
  518 genes selected
  unsup done
  binary done
  multiclass done
  sil=0.006  mc_auc=0.954  (11s)

── svd_top30 ──
  492 genes selected
  unsup done
  binary done
  multiclass done
  sil=0.009  mc_auc=0.963  (11s)

All selectors done.


In [51]:
# ── Global summary: 방법 간 비교 ─────────────────────────────────
global_summary = pd.DataFrame([
    dict(method      = k,
         n_genes     = v['n_genes'],
         macro_sil   = v['macro_sil'],
         macro_mc_auc= v['macro_mc'],
         mean_lr_auc = v['per_pheno']['auc_logreg'].mean(),
         mean_rf_auc = v['per_pheno']['auc_rf'].mean(),
         elapsed_s   = int(v['t']))
    for k, v in all_results.items()
]).set_index('method').sort_values('macro_mc_auc', ascending=False).round(3)

print('══ Global Summary ══')
display(global_summary)

# ── Per-phenotype detail: 최고 method ────────────────────────────
best = global_summary.index[0]
display(all_results[best]['per_pheno'].round(3))

══ Global Summary ══


,n_genes,macro_sil,macro_mc_auc,mean_lr_auc,mean_rf_auc,elapsed_s
method,,,,,,
svd_top30,492,0.009,0.963,0.975,0.944,10
effect_size_top30,518,0.006,0.954,0.971,0.932,11
proportion_top30,537,-0.005,0.948,0.969,0.938,11


,n,silhouette,knn_purity,auc_logreg,auc_rf,auc_multiclass
phenotype,,,,,,
CDCS,224,0.081,0.863,1.000,0.998,1.000
Colorectal Cancer,41,-0.067,0.314,0.996,0.973,0.956
Esophagus Cancer,27,-0.003,0.195,0.962,0.948,0.963
HIV,13,-0.101,0.077,0.893,0.790,0.884
HIV + Tuberculosis,11,-0.012,0.103,0.976,0.936,0.929
ICI-m,11,-0.051,0.339,1.000,1.000,0.997
ICI-treated Cancer,11,0.218,0.521,1.000,1.000,0.998
Liver Cancer,48,-0.103,0.232,0.984,0.939,0.947
Liver Cirrhosis,5,-0.122,0.027,0.993,0.999,0.986


In [55]:
def _roc_shadow(ax, roc_d, color, label, auc, lw=1.5, alpha=0.18):
    """Per-fold ROC curves를 평균 ± std shadow로 표시."""
    if not roc_d or not roc_d.get('fprs'):
        ax.text(0.5, 0.5, 'N/A', ha='center', va='center',
                transform=ax.transAxes, color='grey', fontsize=8)
        return
    tprs = [np.interp(BASE_FPR, f, t)
            for f, t in zip(roc_d['fprs'], roc_d['tprs'])]
    mean_t = np.mean(tprs, axis=0)
    std_t  = np.std(tprs,  axis=0)
    auc_str = f'{auc:.2f}' if not np.isnan(auc) else '—'
    ax.plot(BASE_FPR, mean_t, color=color, lw=lw, label=f'{label} AUC={auc_str}')
    ax.fill_between(BASE_FPR,
                    np.clip(mean_t - std_t, 0, 1),
                    np.clip(mean_t + std_t, 0, 1),
                    alpha=alpha, color=color)


def plot_roc_curves(method_name, save=True):
    res    = all_results[method_name]
    pp     = res['per_pheno']
    phenos = sorted(np.unique(dis_pheno))
    ncols  = 5
    nrows  = (len(phenos) + ncols - 1) // ncols

    def _setup_axes(axes):
        for ax in axes:
            ax.plot([0, 1], [0, 1], 'k--', lw=0.7, alpha=0.35)
            ax.set_xlim(-0.05, 1.05); ax.set_ylim(-0.05, 1.05)
            ax.set_xlabel('FPR', fontsize=6); ax.set_ylabel('TPR', fontsize=6)
            ax.tick_params(labelsize=5)
            ax.legend(fontsize=5.5, frameon=False, loc='lower right')

    # ── (1) Binary: HC vs Disease ─────────────────────────────────
    fig, axes = plt.subplots(nrows, ncols, figsize=(ncols * 3.8, nrows * 3.4))
    axes_flat = axes.flatten()
    for ax, ph in zip(axes_flat, phenos):
        rb  = res['roc_bin'].get(ph)
        n   = (dis_pheno == ph).sum()
        auc_lr = pp.loc[ph, 'auc_logreg'] if ph in pp.index else np.nan
        auc_rf = pp.loc[ph, 'auc_rf']     if ph in pp.index else np.nan
        if rb is None:
            ax.text(0.5, 0.5, f'n={n} < {CV}\n(excluded)',
                    ha='center', va='center', transform=ax.transAxes,
                    fontsize=8, color='grey')
            ax.set_title(f'{ph}  (n={n})', fontsize=7, fontweight='bold')
            ax.axis('off'); continue
        _roc_shadow(ax, rb.get('lr'), '#377eb8', 'LogReg', auc_lr)
        _roc_shadow(ax, rb.get('rf'), '#e41a1c', 'RF',     auc_rf)
        ax.set_title(f'{ph}  (n={n})', fontsize=7, fontweight='bold')
    _setup_axes([axes_flat[i] for i, ph in enumerate(phenos)
                 if res['roc_bin'].get(ph) is not None])
    for ax in axes_flat[len(phenos):]: ax.axis('off')
    fig.suptitle(
        f'HC vs Disease ROC — {method_name}  ({CV}-fold, shadow = ±1 SD)',
        fontsize=11, fontweight='bold', y=1.01,
    )
    plt.tight_layout()
    if save:
        plt.savefig(FIG_DIR / f'roc_binary_{method_name}.png', bbox_inches='tight', dpi=150)
    plt.show()

    # ── (2) Multiclass OVR ────────────────────────────────────────
    fig, axes = plt.subplots(nrows, ncols, figsize=(ncols * 3.8, nrows * 3.4))
    axes_flat = axes.flatten()
    for ax, ph in zip(axes_flat, phenos):
        rm  = res['roc_mc'].get(ph, {})
        n   = (dis_pheno == ph).sum()
        auc = pp.loc[ph, 'auc_multiclass'] if ph in pp.index else np.nan
        _roc_shadow(ax, rm, '#4daf4a', 'OVR LogReg', auc)
        ax.plot([0,1],[0,1],'k--',lw=0.7,alpha=0.35)
        ax.set_xlim(-0.05,1.05); ax.set_ylim(-0.05,1.05)
        ax.set_title(f'{ph}  (n={n})', fontsize=7, fontweight='bold')
        ax.set_xlabel('FPR', fontsize=6); ax.set_ylabel('TPR', fontsize=6)
        ax.tick_params(labelsize=5)
        ax.legend(fontsize=5.5, frameon=False, loc='lower right')
    for ax in axes_flat[len(phenos):]: ax.axis('off')
    fig.suptitle(
        f'Multiclass OVR ROC — {method_name}  ({CV}-fold, shadow = ±1 SD)',
        fontsize=11, fontweight='bold', y=1.01,
    )
    plt.tight_layout()
    if save:
        plt.savefig(FIG_DIR / f'roc_multiclass_{method_name}.png', bbox_inches='tight', dpi=150)
    plt.show()

In [56]:
def visualize(method_name, save=True):
    res  = all_results[method_name]
    idx  = _idx(res['genes'])
    X    = Z_dis[:, idx]

    unique_phenos = sorted(np.unique(dis_pheno))
    p2c = dict(zip(unique_phenos, sns.color_palette('tab20', len(unique_phenos))))
    rc  = [p2c[p] for p in dis_pheno]

    # ── Clustermap ─────────────────────────────────────────────
    pivot = pd.DataFrame(X, index=dis_names, columns=res['genes'])
    g = sns.clustermap(
        pivot, method='ward', metric='euclidean',
        row_colors=rc, col_cluster=True, row_cluster=True,
        cmap='RdBu_r', vmin=-6, vmax=6, center=0,
        xticklabels=False, yticklabels=False,
        figsize=(20, 13), linewidths=0, rasterized=True,
        cbar_kws=dict(label='Z-score', shrink=0.35, ticks=[-6,-3,0,3,6]),
    )
    patches = [Patch(color=p2c[p], label=p) for p in unique_phenos]
    g.ax_heatmap.legend(handles=patches, loc='upper left',
                        bbox_to_anchor=(1.25, 1.0), frameon=False,
                        fontsize=7, title='Phenotype', title_fontsize=8)
    g.fig.suptitle(f'Heatmap — {method_name}  ({len(res["genes"])} genes)',
                   y=1.01, fontsize=11, fontweight='bold')
    if save:
        plt.savefig(FIG_DIR / f'heatmap_{method_name}.png', bbox_inches='tight', dpi=150)
    plt.show()

    # ── UMAP ───────────────────────────────────────────────────
    print('Computing UMAP ...')
    emb = UMAP(n_components=2, metric='euclidean', n_neighbors=15,
               min_dist=0.2, random_state=42).fit_transform(X)
    fig, ax = plt.subplots(figsize=(9, 7))
    for ph in unique_phenos:
        m = dis_pheno == ph
        ax.scatter(emb[m,0], emb[m,1], c=[p2c[ph]], s=12,
                   alpha=0.75, label=ph, edgecolors='none')
    ax.legend(fontsize=7, frameon=False, bbox_to_anchor=(1.01, 1), loc='upper left',
              title='Phenotype', title_fontsize=8)
    ax.set(title=f'UMAP — {method_name}  ({len(res["genes"])} genes)',
           xlabel='UMAP 1', ylabel='UMAP 2')
    plt.tight_layout()
    if save:
        plt.savefig(FIG_DIR / f'umap_{method_name}.png', bbox_inches='tight', dpi=150)
    plt.show()


In [ ]:
for key, item in SELECTORS.items():
    plot_roc_curves(key)
    visualize(key)